# BiLSTM + attention (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_bilstm` (word indices + sequence lengths).

**Run modes** (set flags in the next code cell): **quick** smoke test (`QUICK_ITERATION = True`), **progress report** (`QUICK_ITERATION = False`, `PROGRESS_REPORT_MODE = True`), or **full** (both `False`).

**Metrics:** Same proposal bundle as other starter notebooks.

In [4]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [5]:
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_bilstm

# ========== Run configuration (edit here) ==========
# Three modes (quick wins if True):
#   1) quick smoke     — QUICK_ITERATION = True
#   2) progress report — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = True
#   3) full              — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = False
QUICK_ITERATION = False
PROGRESS_REPORT_MODE = True

LR = 1e-3

# --- Quick smoke-test (used only if QUICK_ITERATION) ---
QUICK_MAX_TRAIN_SAMPLES = 2048
QUICK_MAX_VAL_SAMPLES = 512
QUICK_MAX_LEN = 64
QUICK_BATCH_SIZE = 128
QUICK_EPOCHS = 1
QUICK_MAX_VOCAB = 3000
QUICK_MIN_FREQ = 1
QUICK_HIDDEN = 64

# --- Progress-report run (used if not quick and PROGRESS_REPORT_MODE) ---
PROGRESS_MAX_TRAIN_SAMPLES = 10_000
PROGRESS_MAX_VAL_SAMPLES = 2_000
PROGRESS_MAX_LEN = 128
PROGRESS_BATCH_SIZE = 64
PROGRESS_EPOCHS = 2
PROGRESS_MAX_VOCAB = 10_000
PROGRESS_MIN_FREQ = 2
PROGRESS_HIDDEN = 128

# --- Full run (both flags False) ---
FULL_MAX_TRAIN_SAMPLES = None
FULL_MAX_VAL_SAMPLES = None
FULL_MAX_LEN = 256
FULL_BATCH_SIZE = 64
FULL_EPOCHS = 1
FULL_MAX_VOCAB = 50_000
FULL_MIN_FREQ = 2
FULL_HIDDEN = 128

if QUICK_ITERATION:
    run_mode = "quick (smoke test)"
    _train_n = QUICK_MAX_TRAIN_SAMPLES
    _val_n = QUICK_MAX_VAL_SAMPLES
    MAX_LEN = QUICK_MAX_LEN
    BATCH_SIZE = QUICK_BATCH_SIZE
    EPOCHS = QUICK_EPOCHS
    MAX_VOCAB = QUICK_MAX_VOCAB
    MIN_FREQ = QUICK_MIN_FREQ
    HIDDEN = QUICK_HIDDEN
elif PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    _train_n = PROGRESS_MAX_TRAIN_SAMPLES
    _val_n = PROGRESS_MAX_VAL_SAMPLES
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
    HIDDEN = PROGRESS_HIDDEN
else:
    run_mode = "full"
    _train_n = FULL_MAX_TRAIN_SAMPLES
    _val_n = FULL_MAX_VAL_SAMPLES
    MAX_LEN = FULL_MAX_LEN
    BATCH_SIZE = FULL_BATCH_SIZE
    EPOCHS = FULL_EPOCHS
    MAX_VOCAB = FULL_MAX_VOCAB
    MIN_FREQ = FULL_MIN_FREQ
    HIDDEN = FULL_HIDDEN

data = preprocess_for_bilstm(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)

n_train, n_val = data.X_train.shape[0], data.X_val.shape[0]
print("=" * 60)
print("CONFIG")
print("  mode:", run_mode)
print("  device:", DEVICE)
print("  train_samples:", n_train, "| val_samples:", n_val)
print("  MAX_LEN:", MAX_LEN, "| vocab_size:", vocab_size, "| HIDDEN:", HIDDEN)
print("  BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS)
print("=" * 60)

Device: cpu
CONFIG
  mode: progress report
  device: cpu
  train_samples: 10000 | val_samples: 2000
  MAX_LEN: 128 | vocab_size: 10000 | HIDDEN: 128
  BATCH_SIZE: 64 | EPOCHS: 2


In [6]:
class AttentionBiLSTM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden: int,
        num_labels: int,
        padding_idx: int = 0,
    ):
        super().__init__()
        self.padding_idx = padding_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden * 2, 1)
        self.fc = nn.Linear(hidden * 2, num_labels)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        lens = lengths.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        scores = self.attn(out).squeeze(-1)
        positions = torch.arange(out.size(1), device=x.device).unsqueeze(0).expand_as(scores)
        mask = positions < lengths.unsqueeze(1).to(x.device)
        scores = scores.masked_fill(~mask, -1e9)
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx = (out * w).sum(dim=1)
        return self.fc(ctx)


model = AttentionBiLSTM(vocab_size, embed_dim, HIDDEN, num_labels).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.length_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(
    torch.tensor(data.X_val, dtype=torch.long),
    torch.tensor(data.length_val, dtype=torch.long),
    torch.tensor(data.y_val, dtype=torch.float32),
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

t0 = time.perf_counter()
model.train()
final_train_loss = 0.0
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    for xb, lenb, yb in train_loader:
        xb, lenb, yb = xb.to(DEVICE), lenb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, lenb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    final_train_loss = epoch_loss / max(n_batches, 1)
train_seconds = time.perf_counter() - t0

model.eval()
val_loss_sum = 0.0
val_batches = 0
with torch.no_grad():
    for xb, lenb, yb in val_loader:
        xb, lenb, yb = xb.to(DEVICE), lenb.to(DEVICE), yb.to(DEVICE)
        val_loss_sum += loss_fn(model(xb, lenb), yb).item()
        val_batches += 1
val_loss = val_loss_sum / max(val_batches, 1)

print(f"Training wall time ({EPOCHS} epoch(s)): {train_seconds:.1f} s")
print(f"Final train loss (last epoch avg): {final_train_loss:.4f} | val loss: {val_loss:.4f}")

Training wall time (2 epoch(s)): 192.5 s
Final train loss (last epoch avg): 0.1158 | val loss: 0.1122


In [7]:
def predict_logits_lengths(model, X_np, len_np, batch_size=512):
    model.eval()
    all_logits = []
    x = torch.tensor(X_np, dtype=torch.long)
    lens = torch.tensor(len_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE), lens[i : i + batch_size].to(DEVICE))
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)


logits_val = predict_logits_lengths(model, data.X_val, data.length_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
pred_val = (prob_val >= 0.5).astype(np.int64)

per_label_df, summary = multilabel_evaluation_report(data.y_val, pred_val, prob_val, LABEL_COLUMNS)
print("Micro / macro F1:", summary)
display(per_label_df)
for name, m in per_label_confusion_matrices(data.y_val, pred_val, LABEL_COLUMNS).items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common with very few epochs)."
        " ROC-AUC may still be > 0.5. Use progress/full mode and more EPOCHS for meaningful F1."
    )

print()
print("--- Progress report (concise) ---")
print(f"  mode: {run_mode}")
print(f"  final_train_loss: {final_train_loss:.4f}")
print(f"  val_loss: {val_loss:.4f}")
print(f"  f1_micro: {summary['f1_micro']:.4f}")
print(f"  f1_macro: {summary['f1_macro']:.4f}")

Micro / macro F1: {'f1_micro': 0.23450586264656617, 'f1_macro': 0.10273771479411285}


,label,precision,recall,f1,roc_auc
0,toxic,0.62069,0.266010,0.372414,0.842659
1,severe_toxic,0.00000,0.000000,0.000000,0.940597
2,obscene,0.35000,0.061404,0.104478,0.853649
3,threat,0.00000,0.000000,0.000000,0.918587
4,insult,0.45000,0.082569,0.139535,0.869144
5,identity_hate,0.00000,0.000000,0.000000,0.805709


toxic 
 [[1764   33]
 [ 149   54]]
severe_toxic 
 [[1975    0]
 [  25    0]]
obscene 
 [[1873   13]
 [ 107    7]]
threat 
 [[1996    0]
 [   4    0]]
insult 
 [[1880   11]
 [ 100    9]]
identity_hate 
 [[1985    0]
 [  15    0]]

--- Proposal summary (record for report / comparison) ---
  device: cpu
  training_time_s: 192.52
  num_parameters: 1237319

--- Progress report (concise) ---
  mode: progress report
  final_train_loss: 0.1158
  val_loss: 0.1122
  f1_micro: 0.2345
  f1_macro: 0.1027


## Notes

- Increase `EPOCHS` and add validation-based **early stopping** for the final report.
- Tune **class weights** in `BCEWithLogitsLoss` (`pos_weight`) for rare heads (`threat`, `identity_hate`).